In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
        .appName("Gold")

        # Delta Lake
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

        # Memory (CRITICAL for 100M+ rows)
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "8g")

        # CPU cores
        .config("spark.driver.cores", "4")

        # Shuffle tuning
        .config("spark.sql.shuffle.partitions", "200")

        # Adaptive query optimization
        .config("spark.sql.adaptive.enabled", "true")

        # Prevent huge driver result crashes
        .config("spark.driver.maxResultSize", "2g")

        # UI tuning
        .config("spark.ui.showConsoleProgress", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [2]:
#importing modules
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import logging
import requests

In [3]:
log_dir = "../logs"
os.makedirs(log_dir, exist_ok=True)

logging.basicConfig(
    filename=os.path.join(log_dir, "03_silver_to_gold.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [4]:
logging.info(f"Processing Silver Layer")

In [5]:
df_trip_data = spark.read\
                        .format("delta") \
                        .load(r"..\02_storage_silver\yellow_taxi")

In [6]:
df_trip_data.count()

51028788

### Dimensional Modelling Part 1 - Create Dimensional Tables

From Data Dictionary

In [7]:
vendor_data = [
    (1, "Creative Mobile Technologies"),
    (2, "Curb Mobility"),
    (6, "Myle Technologies"),
    (7, "Helix")
]

dim_vendor = spark.createDataFrame(vendor_data, ["vendor_id", "vendor_name"])

In [8]:
rate_data = [
    (1, "Standard"),
    (2, "JFK"),
    (3, "Newark"),
    (4, "Nassau/Westchester"),
    (5, "Negotiated fare"),
    (6, "Group ride"),
    (99, "Unknown")
]

dim_rate = spark.createDataFrame(rate_data, ["rate_code_id", "rate_name"])

In [9]:
payment_data = [
    (0, "Flex Fare"),
    (1, "Credit Card"),
    (2, "Cash"),
    (3, "No charge"),
    (4, "Dispute"),
    (5, "Unknown"),
    (6, "Voided trip")
]

dim_payment = spark.createDataFrame(payment_data, ["payment_type", "payment_name"])

In [10]:
store_flag_data = [
    ("Y", "Stored and forwarded"),
    ("N", "Sent directly")
]

dim_store_flag = spark.createDataFrame(
    store_flag_data,
    ["store_and_fwd_flag", "store_flag_desc"]
)

Trip Zone Lookup

In [11]:
url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
local_path = r"..\00_storage_raw\taxi_zone_lookup\taxi_zone_lookup.csv"
os.makedirs(os.path.dirname(local_path), exist_ok=True)

if os.path.exists(local_path):
    logging.info("taxi_zone_lookup.csv already exists. Skipping download.")
else:
    logging.info("taxi_zone_lookup.csv not found. Downloading...")

    response = requests.get(url)

    if response.status_code == 200:
        with open(local_path, "wb") as f:
            f.write(response.content)

        logging.info("Location dimension pulled from source API")
    else:
        logging.error(f"Failed to download file. Status code: {response.status_code}")

In [12]:
dim_location = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(local_path)
)
dim_location.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [13]:
dim_location = dim_location.select(
    col("LocationID").alias("location_id"),
    col("Borough").alias("borough"),
    col("Zone").alias("zone"),
    col("service_zone")
)

Write to dim folder

In [14]:
def write_dim(df, name):
    (
        df
        .coalesce(1)
        .write
        .format("delta")
        .mode("overwrite")
        .save(fr"..\03_storage_gold\dimensions\{name}")
    )
    logging.info(f"{name} dimension written to Gold layer")

write_dim(dim_vendor, "dim_vendor")
write_dim(dim_payment, "dim_payment_type")
write_dim(dim_rate, "dim_rate_code")
write_dim(dim_store_flag, "dim_store_forward_flag")
write_dim(dim_location, "dim_location")

### Dimensional Modelling Part 2- Build Fact Tables

Tables for dashboard

In [15]:
fact_trip_yearly_summary = (
    df_trip_data
    .groupBy("trip_year")
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_trip_value"),

        sum("trip_distance").alias("total_distance"),
        avg("trip_distance").alias("avg_distance"),

        avg("fare_per_mile").alias("avg_fare_per_mile"),

        avg("passenger_count").alias("avg_passengers"),

        sum("airport_fee").alias("total_airport_fee"),
        sum("congestion_surcharge").alias("total_congestion_fee"),
        sum("cbd_congestion_fee").alias("total_cbd_fee")
    )
)

fact_trip_yearly_summary.show()

+---------+-----------+--------------------+------------------+-------------------+------------------+------------------+------------------+-----------------+--------------------+-------------+
|trip_year|total_trips|       total_revenue|    avg_trip_value|     total_distance|      avg_distance| avg_fare_per_mile|    avg_passengers|total_airport_fee|total_congestion_fee|total_cbd_fee|
+---------+-----------+--------------------+------------------+-------------------+------------------+------------------+------------------+-----------------+--------------------+-------------+
|     2025|   41129122| 1.208104239360344E9|29.373450747631907|1.481870493798979E8|3.6029713782827137|7.7092833075769205|1.2400011602484489|       5104215.75|       7.515987875E7|2.272175075E7|
|     2026|    9899666|3.0271711409006894E8| 30.57851791061122|3.578222407998252E7| 3.614488012018034| 8.327913778102621|1.1773738629161832|        1131372.0|         1.6069606E7|    5321448.0|
+---------+-----------+-------

In [16]:
fact_payment_summary = (
    df_trip_data
    .groupBy("trip_year", "payment_type")
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_trip_value"),

        avg("fare_per_mile").alias("avg_fare_per_mile")
    )
)

fact_payment_summary.show()

+---------+------------+-----------+--------------------+------------------+-----------------+
|trip_year|payment_type|total_trips|       total_revenue|    avg_trip_value|avg_fare_per_mile|
+---------+------------+-----------+--------------------+------------------+-----------------+
|     2025|           2|    3792900|  9.76513561400049E7|25.745829349575498| 7.97872628068241|
|     2025|           3|      64105|   1595860.920000003|24.894484361594305|8.179298182669045|
|     2025|           1|   28923274|   8.8395613649056E8|30.562104984745506|7.757414985934982|
|     2025|           0|    8304282|2.2362928609000027E8|26.929394508760694|7.413865362471941|
|     2025|           4|      44560|   1271527.730000004|28.535182450628454|7.911486535008985|
|     2025|           5|          1|               71.99|             71.99|             5.18|
|     2026|           1|    6415129|  1.93273302980058E8| 30.12773445086732|7.819645541028829|
|     2026|           2|     791883| 2.05064204200

In [17]:
fact_location_summary = (
    df_trip_data
    .groupBy("trip_year", "pu_location_id")
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),

        avg("total_amount").alias("avg_trip_value"),
        avg("trip_distance").alias("avg_distance"),

        avg("fare_per_mile").alias("avg_fare_per_mile")
    )
)

fact_location_summary.show()

+---------+--------------+-----------+--------------------+------------------+------------------+------------------+
|trip_year|pu_location_id|total_trips|       total_revenue|    avg_trip_value|      avg_distance| avg_fare_per_mile|
+---------+--------------+-----------+--------------------+------------------+------------------+------------------+
|     2025|           237|    1825991| 3.998155896999814E7|21.895813818358437|1.9015290546339223| 8.765110972617082|
|     2025|           163|    1067495|2.8246326950000823E7|26.460383374161772|2.5278945381477356| 8.743829966416829|
|     2025|            10|      19576|  1105724.4700000023| 56.48367746219873|10.001103391908458|5.3614997956681645|
|     2025|            52|      13503|           428231.14| 31.71377767903429| 5.259959268310749| 6.268310745760201|
|     2025|            59|        238|             6845.01|28.760546218487395| 6.039915966386553| 5.407521008403362|
|     2025|           214|        290|  12129.260000000002|41.82

In [18]:
df_with_distance_bucket = (
    df_trip_data
    .withColumn(
        "distance_bucket",

        when(col("trip_distance") < 2, "short")
        .when(col("trip_distance") < 10, "medium")
        .otherwise("long")
    )
)

fact_distance_summary = (
    df_with_distance_bucket
    .groupBy("trip_year", "distance_bucket")
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),

        avg("total_amount").alias("avg_trip_value"),

        avg("fare_per_mile").alias("avg_fare_per_mile"),

        avg("trip_distance").alias("avg_distance")
    )
)

fact_distance_summary.show()

+---------+---------------+-----------+--------------------+------------------+------------------+------------------+
|trip_year|distance_bucket|total_trips|       total_revenue|    avg_trip_value| avg_fare_per_mile|      avg_distance|
+---------+---------------+-----------+--------------------+------------------+------------------+------------------+
|     2025|         medium|   17029353| 5.532809872499796E8| 32.48984193644818| 6.185765463313896| 4.150364602810574|
|     2025|          short|   20713789| 3.788116341495554E8|18.287896731474643| 9.558730880662978|1.2046558217813825|
|     2025|           long|    3385980| 2.760116179601953E8| 81.51602134690556| 4.057594985794312|15.521662552643061|
|     2026|         medium|    4169315|1.4544010737004519E8| 34.88345384554661|6.7660154653690645| 4.226786090758375|
|     2026|           long|     794629| 6.263493906006246E7| 78.82287087441115| 4.012991119125901|15.414806331004419|
|     2026|          short|    4935722| 9.46420676600494

In [19]:
fact_trip_time_summary = (
    df_trip_data
    .groupBy(
        "trip_year",
        "pickup_month",
        "pickup_day",
        "pickup_hour"
    )
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_trip_value"),

        avg("trip_distance").alias("avg_distance"),

        avg("fare_per_mile").alias("avg_fare_per_mile"),

        avg("passenger_count").alias("avg_passengers")
    )
)

fact_trip_time_summary.show()

+---------+------------+----------+-----------+-----------+------------------+------------------+------------------+------------------+------------------+
|trip_year|pickup_month|pickup_day|pickup_hour|total_trips|     total_revenue|    avg_trip_value|      avg_distance| avg_fare_per_mile|    avg_passengers|
+---------+------------+----------+-----------+-----------+------------------+------------------+------------------+------------------+------------------+
|     2025|          12|         7|         23|      32430|1085549.0300000003| 33.47360561208758|3.5367412889299947|   9.6845226641998| 1.253654024051804|
|     2025|           2|         3|         23|      11595|         339530.75| 29.28251401466149| 4.107667097887031| 6.141713669685211|1.2011211729193618|
|     2025|           1|         7|          3|       6399|151146.39999999985| 23.62031567432409|3.0780012501953453| 6.147949679637452| 1.303328645100797|
|     2025|           1|         7|          8|       7281|195045.0900

In [20]:
fact_vendor_summary = (
    df_trip_data
    .groupBy(
        "trip_year",
        "vendor_id"
    )
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),

        avg("total_amount").alias("avg_trip_value"),

        avg("trip_distance").alias("avg_distance"),

        avg("fare_per_mile").alias("avg_fare_per_mile"),

        avg("passenger_count").alias("avg_passengers"),

        sum("airport_fee").alias("total_airport_fee"),

        sum("congestion_surcharge").alias("total_congestion_fee"),

        sum("cbd_congestion_fee").alias("total_cbd_fee")
    )
)

fact_vendor_summary.show()

+---------+---------+-----------+--------------------+------------------+------------------+------------------+------------------+-----------------+--------------------+-------------+
|trip_year|vendor_id|total_trips|       total_revenue|    avg_trip_value|      avg_distance| avg_fare_per_mile|    avg_passengers|total_airport_fee|total_congestion_fee|total_cbd_fee|
+---------+---------+-----------+--------------------+------------------+------------------+------------------+------------------+-----------------+--------------------+-------------+
|     2025|        2|   32248428| 9.584045318504713E8|29.719418628730406|3.5978367757942147| 7.742518139177134|1.2686598242866287|        4327989.0|       5.961818825E7|1.818548075E7|
|     2025|        6|       1688|   58593.01000000002| 34.71149881516589|  9.73664691943128|1.0870023696682465|               1.0|              0.0|                 0.0|          0.0|
|     2025|        1|    8879006|2.4964111450002164E8|28.115885325454407| 3.6204

Write to Disk

In [21]:
def write_fact(df, name):
    (
        df
        .coalesce(1)
        .write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .save(fr"..\03_storage_gold\facts\{name}")
    )
    logging.info(f"{name}  written to Gold layer")

In [22]:
write_fact(fact_trip_yearly_summary, "fact_trip_yearly_summary")
write_fact(fact_payment_summary, "fact_payment_summary")
write_fact(fact_location_summary, "fact_location_summary")
write_fact(fact_distance_summary, "fact_distance_summary")
write_fact(fact_trip_time_summary, "fact_trip_time_summary")
write_fact(fact_vendor_summary, "fact_vendor_summary")